# Train XGBoost Model Using SPARCS 2024

This notebook trains an XGBoost model to predict prolonged length of stay using the 2024 SPARCS inpatient discharge dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# xgboost - for training tree-based gradient boosting model
# joblib - for save the pipeline completely
# shap - for XAI explanation use

!pip install -q xgboost shap joblib

In [ ]:
# Import Libraries

import os
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

In [ ]:
BASE_DIR = "/content/drive/MyDrive/FYP/SPARCS"

PROCESSED_PATH = f"{BASE_DIR}/processed/sparcs_2024_processed.csv"
SPLIT_DIR = f"{BASE_DIR}/splits"
MODEL_DIR = f"{BASE_DIR}/models"
RESULT_DIR = f"{BASE_DIR}/results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Load Processed Dataset and Split Files

df = pd.read_csv(PROCESSED_PATH)

train_idx = np.load(f"{SPLIT_DIR}/train_idx.npy")
val_idx = np.load(f"{SPLIT_DIR}/val_idx.npy")
test_idx = np.load(f"{SPLIT_DIR}/test_idx.npy")

feature_columns = joblib.load(f"{SPLIT_DIR}/feature_columns.joblib")
feature_schema = joblib.load(f"{SPLIT_DIR}/feature_schema.joblib")

print("Dataset shape:", df.shape)
print("Number of selected features:", len(feature_columns))
print(feature_columns)

/tmp/ipykernel_25398/2087352371.py:3: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PROCESSED_PATH)


Dataset shape: (2196737, 35)
Number of selected features: 13
['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']


In [ ]:
# Prepare X and y
# X represents the selected features that staff will later input into the system
# y indicates whether the LOS has been prolonged

target_col = "prolonged_los"

X = df[feature_columns].copy()
y = df[target_col].copy()

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

X_val = X.iloc[val_idx]
y_val = y.iloc[val_idx]

X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1405911, 13) (1405911,)
Validation: (351478, 13) (351478,)
Test: (439348, 13) (439348,)


In [ ]:
# Check Class Distribution
# Here for look at the prolonged LOS ratio
# If the positive rate is, for example, 0.20, then the PR-AUC baseline is 0.20.
# Then the PR-AUC must significantly exceed this baseline for the model to be considered useful.

class_distribution = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    "positive_count": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "total_count": [len(y_train), len(y_val), len(y_test)]
})

class_distribution

,split,positive_rate,positive_count,total_count
0,train,0.241135,339015,1405911
1,validation,0.241136,84754,351478
2,test,0.241137,105943,439348


In [ ]:
# Identify Categorical and Numeric Columns
# The SPARCS feature set mostly consists of categorical dropdown fields
# XGBoost needs to convert string categories into numeric representations
# So use One-Hot Encoding

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical colums:", cat_cols)
print("Numeric columns:", num_cols)

Categorical colums: ['Age Group', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'CCSR Diagnosis Description', 'CCSR Procedure Description', 'APR DRG Description', 'APR MDC Description', 'APR Severity of Illness Description', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Emergency Department Indicator']
Numeric columns: []


In [ ]:
# Build Preprocessing Pipeline
# Missing values are imputed only using training data.
# Unknown categories in validation/test/deployment are ignored safely

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [ ]:
# Handle Class Imbalance
# If the prolonged LOS is a minority class, the model may be biased towards predicting non-prolonged LOS.
# The 'scale_pos_weight' property makes XGBoost place greater emphasis on the prolonged LOS class.

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

scale_pos_weight = negative_count / positive_count

print("Positive count:", positive_count)
print("Negative count:", negative_count)
print("Scale pos weight:", scale_pos_weight)

Positive count: 339015
Negative count: 1066896
Scale pos weight: 3.1470465908588117


In [ ]:
# Train XGBoost Model

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

xgb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  []),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Age Group', 'Gender',
                                                   'Race', 'Ethnicity',
                                                   'Type of Admission',
                                                   'CCSR Diagno...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [ ]:
# Evaluation Function
# ROC-AUC: ranking ability
# PR-AUC: performance on positive class under imbalance
# Recall: ability to detect prolonged LOS cases
# F1: balance of precision and recall

def evaluate_binary_classifier(model, X_data, y_true, split_name, threshold=0.5):
  y_prob = model.predict_proba(X_data)[:, 1]
  y_pred = (y_prob >= threshold).astype(int)

  metrics = {
    "model": "XGBoost",
    "dataset": "SPARCS 2024",
    "split": split_name,
    "threshold": threshold,
    "roc_auc": roc_auc_score(y_true, y_prob),
    "pr_auc": average_precision_score(y_true, y_prob),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0)
  }

  return metrics, y_prob, y_pred

In [ ]:
# Validate Model

val_metrics, y_prob_val, y_pred_val = evaluate_binary_classifier(
    xgb_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=0.5
)

pd.DataFrame([val_metrics])

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,XGBoost,SPARCS 2024,validation,0.5,0.866515,0.680567,0.781844,0.532335,0.784459,0.63426


In [ ]:
print(confusion_matrix(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

[[208315  58409]
 [ 18268  66486]]
              precision    recall  f1-score   support

           0       0.92      0.78      0.84    266724
           1       0.53      0.78      0.63     84754

    accuracy                           0.78    351478
   macro avg       0.73      0.78      0.74    351478
weighted avg       0.83      0.78      0.79    351478



In [ ]:
# Threshold Tuning

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    metrics, _, _ = evaluate_binary_classifier(
        xgb_pipeline,
        X_val,
        y_val,
        "validation",
        threshold=threshold
    )
    threshold_results.append(metrics)

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(by="f1", ascending=False).head(10)

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
8,XGBoost,SPARCS 2024,validation,0.60,0.866515,0.680567,0.809664,0.589169,0.695979,0.638135
7,XGBoost,SPARCS 2024,validation,0.55,0.866515,0.680567,0.799088,0.564512,0.729842,0.636618
6,XGBoost,SPARCS 2024,validation,0.50,0.866515,0.680567,0.781844,0.532335,0.784459,0.634260
9,XGBoost,SPARCS 2024,validation,0.65,0.866515,0.680567,0.819627,0.620750,0.647710,0.633943
5,XGBoost,SPARCS 2024,validation,0.45,0.866515,0.680567,0.756986,0.497669,0.831135,0.622560
10,XGBoost,SPARCS 2024,validation,0.70,0.866515,0.680567,0.828151,0.670719,0.564445,0.613010
4,XGBoost,SPARCS 2024,validation,0.40,0.866515,0.680567,0.723638,0.461418,0.873528,0.603862
3,XGBoost,SPARCS 2024,validation,0.35,0.866515,0.680567,0.680819,0.424558,0.910694,0.579130
11,XGBoost,SPARCS 2024,validation,0.75,0.866515,0.680567,0.828772,0.722382,0.470869,0.570118
2,XGBoost,SPARCS 2024,validation,0.30,0.866515,0.680567,0.626702,0.387273,0.941466,0.548798


# Select Deployment Threshold

The threshold with the best validation F1-score is identified for reference.  
However, the final deployment threshold should also consider the healthcare screening objective, where missing prolonged LOS cases is more critical than flagging some false positives.

In [ ]:
best_f1_threshold_row = threshold_df.sort_values(by="f1", ascending=False).iloc[0]
best_f1_threshold = float(best_f1_threshold_row["threshold"])

best_f1_threshold_row

,8
model,XGBoost
dataset,SPARCS 2024
split,validation
threshold,0.6
roc_auc,0.866515
pr_auc,0.680567
accuracy,0.809664
precision,0.589169
recall,0.695979
f1,0.638135


In [ ]:
candidate_thresholds = threshold_df[threshold_df["recall"] >= 0.70]

if len(candidate_thresholds) > 0:
    deployment_threshold_row = candidate_thresholds.sort_values(
        by=["f1", "precision"],
        ascending=False
    ).iloc[0]
else:
    deployment_threshold_row = best_f1_threshold_row

deployment_threshold = float(deployment_threshold_row["threshold"])

deployment_threshold_row

,7
model,XGBoost
dataset,SPARCS 2024
split,validation
threshold,0.55
roc_auc,0.866515
pr_auc,0.680567
accuracy,0.799088
precision,0.564512
recall,0.729842
f1,0.636618


In [ ]:
# Final Validation and Test Evaluation
# Test set is final unbiased evaluation


val_metrics_deploy, y_prob_val, y_pred_val = evaluate_binary_classifier(
    xgb_pipeline,
    X_val,
    y_val,
    "validation",
    threshold=deployment_threshold
)

test_metrics_deploy, y_prob_test, y_pred_test = evaluate_binary_classifier(
    xgb_pipeline,
    X_test,
    y_test,
    "test",
    threshold=deployment_threshold
)

metrics_df = pd.DataFrame([val_metrics_deploy, test_metrics_deploy])
metrics_df

,model,dataset,split,threshold,roc_auc,pr_auc,accuracy,precision,recall,f1
0,XGBoost,SPARCS 2024,validation,0.55,0.866515,0.680567,0.799088,0.564512,0.729842,0.636618
1,XGBoost,SPARCS 2024,test,0.55,0.865857,0.679612,0.798358,0.563394,0.727797,0.635129


In [ ]:
print("Validation confusion matrix")
print(confusion_matrix(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

print("Test confusion matrix")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

Validation confusion matrix
[[219005  47719]
 [ 22897  61857]]
              precision    recall  f1-score   support

           0       0.91      0.82      0.86    266724
           1       0.56      0.73      0.64     84754

    accuracy                           0.80    351478
   macro avg       0.73      0.78      0.75    351478
weighted avg       0.82      0.80      0.81    351478

Test confusion matrix
[[273652  59753]
 [ 28838  77105]]
              precision    recall  f1-score   support

           0       0.90      0.82      0.86    333405
           1       0.56      0.73      0.64    105943

    accuracy                           0.80    439348
   macro avg       0.73      0.77      0.75    439348
weighted avg       0.82      0.80      0.81    439348



In [ ]:
# Summary Check

positive_baseline = y_train.mean()

summary_check = pd.DataFrame([
    {
        "metric": "Positive baseline",
        "value": positive_baseline,
        "requirement": "PR-AUC should be higher than this"
    },
    {
        "metric": "Validation ROC-AUC",
        "value": val_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Validation PR-AUC",
        "value": val_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Validation Recall",
        "value": val_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    },
    {
        "metric": "Test ROC-AUC",
        "value": test_metrics_deploy["roc_auc"],
        "requirement": ">= 0.70"
    },
    {
        "metric": "Test PR-AUC",
        "value": test_metrics_deploy["pr_auc"],
        "requirement": "> positive baseline"
    },
    {
        "metric": "Test Recall",
        "value": test_metrics_deploy["recall"],
        "requirement": ">= 0.65 preferred"
    }
])

summary_check

,metric,value,requirement
0,Positive baseline,0.241135,PR-AUC should be higher than this
1,Validation ROC-AUC,0.866515,>= 0.70
2,Validation PR-AUC,0.680567,> positive baseline
3,Validation Recall,0.729842,>= 0.65 preferred
4,Test ROC-AUC,0.865857,>= 0.70
5,Test PR-AUC,0.679612,> positive baseline
6,Test Recall,0.727797,>= 0.65 preferred


In [ ]:
# Risk Level Mapping

def risk_level(probability):
    if probability < 0.33:
        return "Low"
    elif probability < 0.66:
        return "Medium"
    else:
        return "High"

In [ ]:
sample_output = pd.DataFrame({
    "predicted_probability": y_prob_test[:10],
    "prolonged_los_prediction": (y_prob_test[:10] >= deployment_threshold).astype(int),
    "risk_level": [risk_level(p) for p in y_prob_test[:10]]
})

sample_output

,predicted_probability,prolonged_los_prediction,risk_level
0,0.469585,0,Medium
1,0.171146,0,Low
2,0.314255,0,Low
3,0.371282,0,Medium
4,0.886522,1,High
5,0.013023,0,Low
6,0.942011,1,High
7,0.838135,1,High
8,0.202837,0,Low
9,0.183252,0,Low


In [ ]:
# Save Model, Metrics, Threshold Results, Metadata

xgb_model_path = f"{MODEL_DIR}/xgboost_sparcs_los_pipeline.joblib"
joblib.dump(xgb_pipeline, xgb_model_path)

metrics_df.to_csv(f"{RESULT_DIR}/xgboost_sparcs_metrics.csv", index=False)
threshold_df.to_csv(f"{RESULT_DIR}/xgboost_sparcs_threshold_tuning.csv", index=False)

metadata = {
    "model_name": "XGBoost",
    "dataset": "SPARCS 2024",
    "target": "prolonged_los",
    "target_definition": "Length of Stay >= 7 days",
    "selected_threshold": deployment_threshold,
    "threshold_selection_reason": "Selected on validation set by prioritizing recall >= 0.70 and then F1-score.",
    "best_f1_threshold": best_f1_threshold,
    "risk_level_thresholds": {
        "low": "<0.33",
        "medium": "0.33-0.66",
        "high": ">=0.66"
    },
    "feature_columns": feature_columns,
    "feature_schema": feature_schema
}

joblib.dump(metadata, f"{MODEL_DIR}/xgboost_sparcs_metadata.joblib")

print("Saved XGBoost pipeline:", xgb_model_path)
print("Saved metrics and metadata.")

Saved XGBoost pipeline: /content/drive/MyDrive/FYP/SPARCS/models/xgboost_sparcs_los_pipeline.joblib
Saved metrics and metadata.


## Summary

An XGBoost model was trained using the selected SPARCS 2024 deployment-friendly features.  
The model excluded payment, cost, facility identifier, hospital geography, and discharge outcome fields to align with the system input form and reduce leakage.  
The model was evaluated using ROC-AUC, PR-AUC, accuracy, precision, recall, and F1-score.  
Threshold tuning was performed on the validation set, and the final threshold was selected by prioritizing recall for prolonged LOS screening.  
The trained pipeline, metrics, threshold tuning results, and metadata were saved for later model comparison and deployment.